# Data processing and panel construction

This notebook constructs the main wide panel used for the project in SQL and GeoPandas.

The structure of the notebook is as follows:
- Load and extract the bicycle site metadata
- Load the bicycle counts data and merge the longitude and latitude.
- Create balanced set from the full set

In [1]:
import duckdb
import geopandas as gpd

## Get metadata

In [2]:
sites_geojson_path = "../data/sites/Bicycle_count_sites.geojson"

gdf_sites = gpd.read_file(sites_geojson_path)

gdf_sites.head()

,OBJECTID,SiteID,Intersection,geometry
0,1,1,"Intersection of Broadway, Lee Street, Quay Str...",POINT (151.20417 -33.88285)
1,2,2,Intersection of Castlereagh Street and Goulbur...,POINT (151.2086 -33.87829)
2,3,3,Intersection of Grosvenor Street and Glouceste...,POINT (151.20578 -33.86336)
3,4,4,Intersection of Elizabeth Street and Park Street,POINT (151.2099 -33.87329)
4,5,5,"Intersection of Elizabeth Street, Wentworth Av...",POINT (151.20908 -33.8798)


In [3]:
# extract lon/lat
gdf_sites["longitude"] = gdf_sites.geometry.x
gdf_sites["latitude"]  = gdf_sites.geometry.y
gdf_sites = gdf_sites[["SiteID", "longitude", "latitude"]].copy()
gdf_sites = gdf_sites.rename(columns={"SiteID": "site_id"})

## Load counts data and merge

In [4]:
# create/load duckdb data
con = duckdb.connect("../data/transport_project.duckdb")

# make sites matadata table for SQL
con.register("bike_sites", gdf_sites)

con.execute("SELECT * FROM bike_sites LIMIT 5").fetch_df()

,site_id,longitude,latitude
0,1,151.204168,-33.882846
1,2,151.208601,-33.878286
2,3,151.205783,-33.863364
3,4,151.209897,-33.873292
4,5,151.209083,-33.879797


In [ ]:
# load csv to table
counts_path = "../data/counts/Bicycle_count_surveys.csv"

con.execute(f"""
    CREATE OR REPLACE TABLE bike_counts AS
    SELECT *
    FROM read_csv('{counts_path}', HEADER=TRUE)
""")

con.execute("SELECT * FROM bike_counts LIMIT 5").fetchdf()

,SiteID,Month,Year,TotalCount,ObjectId2,Time_0600,Time_0700,Time_0800,Time_1600,Time_1700,Time_1800
0,51,March,2010,263,1,12,45,56,27,56,67
1,1,October,2015,383,2,37,69,100,47,68,62
2,52,March,2010,136,3,7,18,31,29,30,21
3,53,March,2010,333,4,25,86,93,15,62,52
4,2,October,2015,447,5,32,75,72,56,114,98


In [6]:
# note inconsistency in month name
con.execute("SELECT DISTINCT Month FROM bike_counts").fetch_df()

,Month
0,October
1,October
2,March


In [7]:
# rename columns, make date, and trim the spaces for month name
con.execute("""
    CREATE OR REPLACE TABLE bike_counts_clean AS
    SELECT
        SiteID AS site_id,
        MAKE_DATE(
            year,
            CASE
                WHEN month = 'March' THEN 3
                WHEN month = 'October' Then 10
                ELSE NULL
            END,
            1
        ) AS survey_date,
        Year AS year, TRIM(Month) AS month,
        TotalCount AS total_count,
        Time_0600 AS time_0600, Time_0700 AS time_0700, Time_0800 AS time_0800,
        Time_1600 AS time_1600, Time_1700 AS time_1700, Time_1800 AS time_1800,
    FROM bike_counts
    ORDER BY site_id, survey_date
""")

con.execute("SELECT * FROM bike_counts_clean LIMIT 5").fetch_df()

,site_id,survey_date,year,month,total_count,time_0600,time_0700,time_0800,time_1600,time_1700,time_1800
0,1,2010-03-01,2010,March,287,24,42,86,44,42,49
1,1,2010-10-01,2010,October,436,56,104,63,66,100,47
2,1,2011-03-01,2011,March,373,26,57,77,80,79,54
3,1,2011-10-01,2011,October,467,29,90,139,75,64,70
4,1,2012-03-01,2012,March,525,41,92,132,90,96,74


In [8]:
# merge on site id
con.execute("""
    CREATE OR REPLACE TABLE bike_panel_wide AS
    SELECT b.*, s.longitude, s.latitude,
    FROM bike_counts_clean AS b
    INNER JOIN bike_sites AS s
        ON b.site_id = s.site_id
""")

con.execute("SELECT * FROM bike_panel_wide LIMIT 5").fetch_df()

,site_id,survey_date,year,month,total_count,time_0600,time_0700,time_0800,time_1600,time_1700,time_1800,longitude,latitude
0,1,2010-03-01,2010,March,287,24,42,86,44,42,49,151.204168,-33.882846
1,1,2010-10-01,2010,October,436,56,104,63,66,100,47,151.204168,-33.882846
2,1,2011-03-01,2011,March,373,26,57,77,80,79,54,151.204168,-33.882846
3,1,2011-10-01,2011,October,467,29,90,139,75,64,70,151.204168,-33.882846
4,1,2012-03-01,2012,March,525,41,92,132,90,96,74,151.204168,-33.882846


## Create balanced panel for project

In [9]:
# create table of site ids with full survey dates (for balanced set)
con.execute("""
    CREATE OR REPLACE TABLE balanced_site_id AS
    SELECT
        site_id, 
        COUNT(DISTINCT survey_date) AS n_surveys
    FROM bike_panel_wide
    GROUP BY site_id
    HAVING COUNT(DISTINCT survey_date) =
        (
        SELECT COUNT(DISTINCT survey_date)
        FROM bike_panel_wide
        )
    ORDER BY site_id
""")

# make balanced panel
con.execute("""
    CREATE OR REPLACE TABLE bike_panel_wide_balanced AS
    SELECT b.*
    FROM bike_panel_wide AS b
    INNER JOIN balanced_site_id AS s
        ON b.site_id = s.site_id
""")

con.execute("SELECT * FROM bike_panel_wide_balanced LIMIT 5").fetch_df()

,site_id,survey_date,year,month,total_count,time_0600,time_0700,time_0800,time_1600,time_1700,time_1800,longitude,latitude
0,2,2010-03-01,2010,March,140,7,12,22,29,32,38,151.208601,-33.878286
1,2,2010-10-01,2010,October,84,5,16,23,12,13,15,151.208601,-33.878286
2,2,2011-03-01,2011,March,106,19,22,12,20,22,11,151.208601,-33.878286
3,2,2011-10-01,2011,October,243,40,45,39,32,53,34,151.208601,-33.878286
4,2,2012-03-01,2012,March,299,12,23,28,48,112,76,151.208601,-33.878286
